# Adaptive Metaheuristics, Hybrid Methods, and Learned Method Selection for Resource Allocation in Software-Defined Vehicles

**Authors:** Mahmoud S. Hamza, Ousman Khan, Alaa Khamis  
**Affiliation:** AI for Smart Mobility Lab, KFUPM  
**Course:** Optimization Algorithms

This notebook accompanies the IEEE conference paper of the same title and contains the full implementation of all proposed methods, benchmark generation, evaluation, and the figures used in the paper.

## Contents

1. **Setup** — imports, constants, environment
2. **Problem formulation helpers** — data loaders, graph utilities, feasibility checker
3. **CSCH-Guided constructive baseline** — feasible-by-construction warm-start
4. **Adaptive bio-inspired metaheuristics** — Simulated Annealing, Tabu Search, Genetic Algorithm
5. **Memetic matheuristic hybrid** — H-Comp and H-Nest
6. **Path-A ILP reference** — exact baseline
7. **Adaptive method selector** — feature extraction and Random Forest selector
8. **Benchmark generation** — synthetic SDV instances
9. **Results** — cross-validated evaluation, tables, and figures
10. **Discussion**

## Reproducibility

All instances are generated synthetically from fixed seeds inside this notebook. No external data download or cloud-storage access is required. The notebook supports two modes via the `QUICK_DEMO` flag:

- `QUICK_DEMO = True` (default): runs a small subset (≈10 minutes on a Colab CPU). Verifies that all methods work and produces a downsized version of every figure and table.
- `QUICK_DEMO = False`: reproduces the full 48-instance benchmark from the paper (≈3–4 hours, including ILP runs).

Gurobi is required for the matheuristic and ILP components. A free academic license is sufficient for the largest instance ($n = 800$).

## 1. Setup

Imports, package installation, and environment configuration. The Gurobi solver is installed via pip; for license activation, follow the [Gurobi Academic License instructions](https://www.gurobi.com/academia/academic-program-and-licenses/).

In [ ]:
# Install Gurobi (no-op if already installed)
!pip install -q gurobipy

import os
import time
import math
import random
from collections import deque
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gurobipy as gp
from gurobipy import GRB

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold

# Reproducibility
np.random.seed(0)
random.seed(0)

# Run mode
QUICK_DEMO = True  # Set False to reproduce the full paper benchmark

# Gurobi environment (uses default license; supply your own if needed)
GRB_ENV = gp.Env(empty=True)
GRB_ENV.setParam("OutputFlag", 0)
GRB_ENV.start()
print("Environment ready.")

### 1.1 Problem constants

These constants define the SDV platform under study (homogeneous VM pool, multi-resource capacity, and the standard 80-VM ceiling). They follow the convention of Pan et al.\ (2022) and Khan & Khamis (2025), referred to as Path~A in the paper.

In [ ]:
# Per-application demand
APP_CORE = 0.05    # cores per application
APP_MEM  = 50.0    # MB per application

# Per-VM capacity
VM_CORE  = 1.0     # cores per VM
VM_MEM   = 96000.0 # MB per VM

# Platform aggregate capacity
MAX_CORE = 80.0
MAX_MEM  = 768000.0

# Derived
MAX_APPS_PER_VM = int(VM_CORE / APP_CORE)   # 20
VM_CAP          = 80                        # active-VM ceiling

## 2. Problem Formulation Helpers

Helpers for instance generation, graph manipulation, and feasibility verification. An *instance* consists of three arrays:

- `safe[a]` — 1 if application $a$ is safety-critical (ASIL A–D), 0 otherwise (QM)
- `dep[a, b]` — 1 if applications $a$ and $b$ are functionally dependent (must share at least one host)
- `conf[a, b]` — 1 if applications $a$ and $b$ conflict (cannot share a host)

In [ ]:
def generate_instance(n_app, seed, p_dep=0.10, p_conf=0.10):
    """Generate a synthetic SDV instance with given size and density.

    Edges are placed only between applications of the same safety class
    (consistent with the benchmark distribution used in the paper).
    """
    np.random.seed(seed)
    random.seed(seed)
    safe = np.random.randint(2, size=n_app)
    dep  = np.zeros((n_app, n_app), dtype=int)
    conf = np.zeros((n_app, n_app), dtype=int)
    for i in range(n_app):
        for j in range(i + 1, n_app):
            if safe[i] == safe[j]:
                r = random.random()
                if r < p_dep:
                    dep[i, j] = 1
                elif r < p_dep + p_conf:
                    conf[i, j] = 1
    return safe.astype(int), dep, conf


def components_from_dep(dep):
    """Return connected components of the dependency graph as lists of indices."""
    n = dep.shape[0]
    adj = [[] for _ in range(n)]
    for i in range(n):
        for j in range(i + 1, n):
            if dep[i, j] == 1 or dep[j, i] == 1:
                adj[i].append(j)
                adj[j].append(i)
    comp_id = [-1] * n
    comps = []
    for i in range(n):
        if comp_id[i] != -1:
            continue
        q = deque([i])
        comp_id[i] = len(comps)
        cur = [i]
        while q:
            u = q.popleft()
            for v in adj[u]:
                if comp_id[v] == -1:
                    comp_id[v] = comp_id[i]
                    q.append(v)
                    cur.append(v)
        comps.append(cur)
    return comps


def greedy_color_with_order(nodes, conf, order=None):
    """Greedy graph coloring of `nodes` on the conflict graph."""
    if not nodes:
        return []
    if order is None:
        seq = nodes[:]
    else:
        rank = {b: i for i, b in enumerate(order)}
        seq = sorted(nodes, key=lambda a: rank.get(a, 10**9))
    colors = []
    for a in seq:
        placed = False
        for col in colors:
            if all(conf[a, b] == 0 and conf[b, a] == 0 for b in col):
                col.append(a)
                placed = True
                break
        if not placed:
            colors.append([a])
    return colors


def bin_by_capacity(app_list):
    """Split `app_list` into VM-sized bins of at most MAX_APPS_PER_VM apps."""
    bins, cur = [], []
    for a in app_list:
        cur.append(a)
        if len(cur) == MAX_APPS_PER_VM:
            bins.append(cur)
            cur = []
    if cur:
        bins.append(cur)
    return bins

### 2.1 Allocation verifier

Before reporting any solution as feasible we run it through an explicit verifier that checks every constraint of the model: redundancy, freedom-from-interference, conflict isolation, dependency, and the hard VM cap.

In [ ]:
def verify_allocation(alloc, safe, dep, conf, *, strict_dependency=False):
    """Return (is_feasible, list_of_violations).

    By default we use the operational dependency interpretation (a dependent
    pair sharing one VM in some component is enough). Pass strict_dependency=True
    to enforce the per-edge interpretation of constraint (C4).
    """
    violations = []
    n = len(safe)

    placements = {a: [] for a in range(n)}
    for vm_id, info in alloc.items():
        for a in info["apps"]:
            placements[a].append(vm_id)

    # Redundancy (C3)
    for a in range(n):
        expected = 2 if safe[a] == 1 else 1
        if len(placements[a]) != expected:
            violations.append(("redundancy", a, len(placements[a]), expected))

    # Freedom from interference (C2)
    for vm_id, info in alloc.items():
        for a in info["apps"]:
            if bool(safe[a]) != info["safety"]:
                violations.append(("FFI", vm_id, a))

    # Conflict isolation (C5)
    for vm_id, info in alloc.items():
        apps = info["apps"]
        for i, a in enumerate(apps):
            for b in apps[i+1:]:
                if conf[a, b] == 1 or conf[b, a] == 1:
                    violations.append(("conflict", vm_id, a, b))

    # Strict dependency (optional, C4)
    if strict_dependency:
        for a in range(n):
            for b in range(a + 1, n):
                if dep[a, b] == 1 or dep[b, a] == 1:
                    if not (set(placements[a]) & set(placements[b])):
                        violations.append(("dep_strict", a, b))

    # Capacity / VM-cap
    if len(alloc) > VM_CAP:
        violations.append(("vm_cap", len(alloc)))
    for vm_id, info in alloc.items():
        if len(info["apps"]) > MAX_APPS_PER_VM:
            violations.append(("over_capacity", vm_id, len(info["apps"])))

    return len(violations) == 0, violations

## 3. CSCH-Guided Constructive Baseline

CSCH-Guided is the constructive heuristic of Khan & Khamis (2025). It processes each dependency component independently: greedy-coloring the conflict graph by a degree-aware ordering, partitioning each color class into VM-sized bins of at most $\lfloor C^{\text{cpu}}_v / r_a^{\text{cpu}} \rfloor = 20$ applications, and replicating the safety class through a second placement pass. The result is feasible by construction and runs in sub-second time at all benchmark sizes. CSCH-Guided is used both as a standalone baseline and as the initial solution for SA, TS, GA, and the memetic hybrid.

In [ ]:
def build_feasible_with_assignment(safe, dep, conf, ohs=None, ohn=None):
    """Construct a feasible allocation using greedy coloring and bin packing.

    `ohs[cid]` and `ohn[cid]` give an optional ordering hint for the safety
    and non-safety nodes of dependency component `cid`. CSCH-Guided uses
    a degree-descending hint; vanilla CSCH passes None.
    """
    comps = components_from_dep(dep)
    allocation = {}
    vm_id = 0
    for cid, C in enumerate(comps):
        has_safe = any(safe[a] == 1 for a in C)
        C_safe = [a for a in C if safe[a] == 1]
        C_non  = [a for a in C if safe[a] == 0]
        os_h = None if ohs is None else ohs.get(cid)
        on_h = None if ohn is None else ohn.get(cid)
        safe_cols = greedy_color_with_order(C_safe, conf, order=os_h)
        non_cols  = greedy_color_with_order(C_non,  conf, order=on_h)
        safe_bins = [bin_by_capacity(col) for col in safe_cols]
        non_bins  = [bin_by_capacity(col) for col in non_cols]
        replicas = 2 if has_safe else 1
        for _ in range(replicas):
            for bins in safe_bins:
                for b in bins:
                    allocation[vm_id] = {"apps": list(b), "safety": True}
                    vm_id += 1
                    if vm_id > VM_CAP:
                        return None, vm_id, "vm_cap_exceeded"
            for bins in non_bins:
                for b in bins:
                    allocation[vm_id] = {"apps": list(b), "safety": False}
                    vm_id += 1
                    if vm_id > VM_CAP:
                        return None, vm_id, "vm_cap_exceeded"

    total = sum(len(info["apps"]) for info in allocation.values())
    if total * APP_CORE > MAX_CORE + 1e-9:
        return None, len(allocation), "platform_core"
    if total * APP_MEM  > MAX_MEM  + 1e-6:
        return None, len(allocation), "platform_mem"
    return allocation, len(allocation), ""


def csch(safe, dep, conf):
    """Vanilla CSCH (no ordering hint)."""
    return build_feasible_with_assignment(safe, dep, conf)


def csch_guided(safe, dep, conf):
    """CSCH-Guided: degree-descending ordering inside each component."""
    comps = components_from_dep(dep)
    deg = np.array(dep.sum(axis=1) + conf.sum(axis=1))
    ohs, ohn = {}, {}
    for cid, C in enumerate(comps):
        Cs = [a for a in C if safe[a] == 1]
        Cn = [a for a in C if safe[a] == 0]
        if Cs:
            ohs[cid] = sorted(Cs, key=lambda a: -deg[a])
        if Cn:
            ohn[cid] = sorted(Cn, key=lambda a: -deg[a])
    return build_feasible_with_assignment(safe, dep, conf, ohs, ohn)

## 4. Adaptive Bio-Inspired Metaheuristics

Three classical metaheuristics adapted to the SDV setting, each starting from the CSCH-Guided allocation. All three minimise the same cost function (number of active VMs) and share the same neighbourhood operators, but differ in their search strategy:

- **Simulated Annealing** — single-solution, stochastic acceptance with adaptive cooling rate
- **Tabu Search** — single-solution, deterministic acceptance with reactive tabu list length
- **Genetic Algorithm** — population-based, component-wise crossover with adaptive mutation rate

### 4.1 Neighbourhood operators

The neighbourhood combines two operators. The *single-app move* relocates one application's placement to a feasible alternative VM. The *empty-smallest-VM move* (a compound operator) picks one of the three least-utilised VMs and tries to redistribute all of its applications to other VMs through a best-fit greedy pass. Without the compound operator, single-app moves alone struggle to reduce the VM count once CSCH-Guided has filled most VMs near capacity.

In [ ]:
def _build_placements(allocation):
    p = {}
    for vm_id, info in allocation.items():
        for a in info["apps"]:
            p.setdefault(a, []).append(vm_id)
    return p


def _renumber(allocation):
    return {i: v for i, (_, v) in enumerate(sorted(allocation.items()))}


def _neighbor_move(allocation, safe, conf, rng):
    """Single-app move: relocate one application to a feasible alternative VM."""
    placements = _build_placements(allocation)
    apps = list(placements.keys())
    if not apps:
        return None
    a = rng.choice(apps)
    src_vm = rng.choice(placements[a])
    a_safety = bool(safe[a] == 1)
    other = set(placements[a]) - {src_vm}
    cands = []
    for vm_id, info in allocation.items():
        if vm_id == src_vm or vm_id in other:        continue
        if info["safety"] != a_safety:               continue
        if len(info["apps"]) >= MAX_APPS_PER_VM:     continue
        if any(conf[a, b] == 1 or conf[b, a] == 1 for b in info["apps"]):
            continue
        cands.append(vm_id)
    cands.append("new")
    dst = rng.choice(cands)
    new = deepcopy(allocation)
    new[src_vm]["apps"].remove(a)
    if dst == "new":
        nid = (max(new.keys()) + 1) if new else 0
        new[nid] = {"apps": [a], "safety": a_safety}
    else:
        new[dst]["apps"].append(a)
    new = {k: v for k, v in new.items() if v["apps"]}
    return _renumber(new)


def _neighbor_empty_vm(allocation, safe, conf, rng, k_smallest=3):
    """Compound move: pick one of the k smallest VMs and redistribute its apps."""
    if len(allocation) <= 1:
        return None
    by_size = sorted(allocation.keys(), key=lambda vm: len(allocation[vm]["apps"]))
    target = rng.choice(by_size[:k_smallest])
    targs = list(allocation[target]["apps"])
    new_alloc = deepcopy(allocation)
    new_alloc[target]["apps"] = []
    pmap = _build_placements(allocation)
    for a in targs:
        a_safety = bool(safe[a] == 1)
        other = set(pmap[a]) - {target}
        cands = []
        for vm_id, info in new_alloc.items():
            if vm_id == target or vm_id in other:    continue
            if info["safety"] != a_safety:           continue
            if len(info["apps"]) >= MAX_APPS_PER_VM: continue
            if any(conf[a, b] == 1 or conf[b, a] == 1 for b in info["apps"]):
                continue
            cands.append((len(info["apps"]), vm_id))
        if not cands:
            return None
        cands.sort(reverse=True)
        chosen = cands[0][1]
        new_alloc[chosen]["apps"].append(a)
    del new_alloc[target]
    return _renumber(new_alloc)

### 4.2 Adaptive Simulated Annealing

The cooling rate $\alpha$ adapts based on the recent acceptance ratio $\rho_t$ measured over a sliding window of $W = 200$ iterations. If $\rho_t > 1.5\,\rho^{\star}$ where $\rho^{\star} = 0.30$ is the target acceptance rate, $\alpha$ is increased toward 1 to cool more slowly; if $\rho_t < 0.5\,\rho^{\star}$, $\alpha$ is decreased to cool faster. This reactive schedule maintains exploration–exploitation balance across the run.

In [ ]:
def simulated_annealing(safe, dep, conf, *,
                        max_iter=4000, T0=2.0, T_min=0.01,
                        cooling_rate=0.998, empty_vm_prob=0.4,
                        adaptive=True, check_window=200,
                        target_accept_rate=0.30, seed=0):
    """Adaptive Simulated Annealing for VM allocation."""
    rng = random.Random(seed)
    current, _, reason = csch_guided(safe, dep, conf)
    if current is None:
        return None, 0, f"csch_failed:{reason}"

    cc = len(current)
    best, bc = deepcopy(current), cc
    T = T0
    cr = cooling_rate
    acc, iw = 0, 0

    for it in range(max_iter):
        if rng.random() < empty_vm_prob:
            n = _neighbor_empty_vm(current, safe, conf, rng)
        else:
            n = _neighbor_move(current, safe, conf, rng)
        if n is None:
            T = max(T_min, T * cr)
            continue

        nc = len(n)
        delta = nc - cc
        if delta <= 0 or rng.random() < math.exp(-delta / T):
            current = n
            cc = nc
            acc += 1
            if cc < bc:
                best, bc = deepcopy(current), cc
        T = max(T_min, T * cr)
        iw += 1

        # Adaptive control: adjust cooling rate based on acceptance ratio
        if adaptive and iw >= check_window:
            rate = acc / iw
            if rate > target_accept_rate * 1.5:
                cr = min(0.9995, cr + 0.001)
            elif rate < target_accept_rate * 0.5:
                cr = max(0.985, cr - 0.002)
            acc, iw = 0, 0

    return best, bc, ""

### 4.3 Adaptive Tabu Search

The tabu list length $L$ follows the Reactive Tabu Search principle (Battiti & Tecchiolli 1994). After 80 iterations without improvement, $L$ is reduced to encourage diversification. After a strict improvement in the best-known allocation, $L$ is occasionally increased to intensify the search. A no-improvement threshold of 400 iterations terminates the run.

In [ ]:
def _neighbor_with_signature(allocation, safe, conf, rng, empty_vm_prob=0.4):
    """Return (neighbor, tabu_signature) for use inside Tabu Search."""
    if rng.random() < empty_vm_prob:
        return _neighbor_empty_vm(allocation, safe, conf, rng), ("empty",)

    placements = _build_placements(allocation)
    apps = list(placements.keys())
    if not apps:
        return None, None
    a = rng.choice(apps)
    src_vm = rng.choice(placements[a])
    a_safety = bool(safe[a] == 1)
    other = set(placements[a]) - {src_vm}
    cands = []
    for vm_id, info in allocation.items():
        if vm_id == src_vm or vm_id in other:        continue
        if info["safety"] != a_safety:               continue
        if len(info["apps"]) >= MAX_APPS_PER_VM:     continue
        if any(conf[a, b] == 1 or conf[b, a] == 1 for b in info["apps"]):
            continue
        cands.append(vm_id)
    cands.append("new")
    dst = rng.choice(cands)
    new = deepcopy(allocation)
    new[src_vm]["apps"].remove(a)
    if dst == "new":
        nid = (max(new.keys()) + 1) if new else 0
        new[nid] = {"apps": [a], "safety": a_safety}
    else:
        new[dst]["apps"].append(a)
    new = _renumber({k: v for k, v in new.items() if v["apps"]})
    sig = ("move", a, frozenset(allocation[src_vm]["apps"]) - {a})
    return new, sig


def tabu_search(safe, dep, conf, *,
                max_iter=2000, tabu_len_init=15,
                neighbors_per_iter=15, empty_vm_prob=0.4,
                adaptive_tabu=True, max_no_improve=400, seed=0):
    """Adaptive Tabu Search for VM allocation."""
    rng = random.Random(seed)
    current, _, reason = csch_guided(safe, dep, conf)
    if current is None:
        return None, 0, f"csch_failed:{reason}"

    cc = len(current)
    best, bc = deepcopy(current), cc
    tabu = deque(maxlen=tabu_len_init)
    tabu_len = tabu_len_init
    nimp = 0

    for it in range(max_iter):
        cands = []
        for _ in range(neighbors_per_iter):
            n, sig = _neighbor_with_signature(current, safe, conf, rng, empty_vm_prob)
            if n is None:
                continue
            nc = len(n)
            is_tabu = sig in tabu
            # Aspiration: accept tabu move only if it strictly improves the best
            if is_tabu and nc >= bc:
                continue
            cands.append((nc, n, sig))

        if not cands:
            continue

        cands.sort(key=lambda x: x[0])
        nc, na, sig = cands[0]
        current = na
        cc = nc
        tabu.append(sig)
        if cc < bc:
            best, bc = deepcopy(current), cc
            nimp = 0
        else:
            nimp += 1

        # Adaptive tabu length
        if adaptive_tabu:
            if nimp > 80 and tabu_len > 5:
                tabu_len = max(5, tabu_len - 2)
                tabu = deque(list(tabu)[-tabu_len:], maxlen=tabu_len)
                nimp = 40
            elif nimp == 0 and tabu_len < 40 and rng.random() < 0.1:
                tabu_len = min(40, tabu_len + 2)
                tabu = deque(list(tabu), maxlen=tabu_len)

        if nimp >= max_no_improve:
            break

    return best, bc, ""

### 4.4 Adaptive Genetic Algorithm

Each individual is a complete VM allocation. The initial population of size 12 is seeded with one CSCH-Guided solution and 11 diversified solutions obtained from short SA runs. Selection uses tournaments of size 3.

**Component-wise crossover.** The benchmark distribution carries no cross-component edges, so a child can inherit each dependency component's allocation entirely from one parent. This crossover is feasibility-preserving by construction and avoids the repair step typical of GA encodings for combinatorial optimisation.

**Adaptive mutation.** The mutation rate $p_{\text{mut}}$ adapts to population diversity, measured as the number of distinct fitness values: it is increased when diversity drops to two or fewer values (signalling premature convergence) and decreased when diversity exceeds half the population size. Elitism preserves the best individual across generations.

In [ ]:
def _crossover_component(parent1, parent2, dep, rng):
    """Component-wise crossover. Each dependency component is inherited
    entirely from one parent. Feasibility-preserving by construction."""
    components = components_from_dep(dep)
    child = {}
    next_vm_id = 0
    for C in components:
        component_set = set(C)
        source = parent1 if rng.random() < 0.5 else parent2
        for vm_id, info in source.items():
            apps_in_comp = [a for a in info["apps"] if a in component_set]
            if apps_in_comp:
                child[next_vm_id] = {"apps": apps_in_comp, "safety": info["safety"]}
                next_vm_id += 1
    return child


def adaptive_genetic_algorithm(safe, dep, conf, *,
                               pop_size=12, generations=25,
                               base_mutation_rate=0.25, tournament_size=3,
                               adaptive=True, seed=0):
    """Adaptive Genetic Algorithm with component-wise crossover."""
    rng = random.Random(seed)

    base, _, _ = csch_guided(safe, dep, conf)
    if base is None:
        return None, 0, "csch_failed"
    population = [deepcopy(base)]
    while len(population) < pop_size:
        sa_seed = rng.randint(0, 10**6)
        alloc, _, _ = simulated_annealing(
            safe, dep, conf,
            max_iter=300, T0=3.0, cooling_rate=0.99,
            empty_vm_prob=0.4, seed=sa_seed)
        if alloc is not None:
            population.append(alloc)

    fitness = [len(ind) for ind in population]
    best_idx = min(range(len(population)), key=lambda i: fitness[i])
    best = deepcopy(population[best_idx])
    best_cost = fitness[best_idx]
    mutation_rate = base_mutation_rate

    for gen in range(generations):
        new_pop = [deepcopy(best)]   # elitism
        while len(new_pop) < pop_size:
            t1 = rng.sample(range(len(population)), tournament_size)
            t2 = rng.sample(range(len(population)), tournament_size)
            p1 = min(t1, key=lambda i: fitness[i])
            p2 = min(t2, key=lambda i: fitness[i])
            child = _crossover_component(population[p1], population[p2], dep, rng)
            if rng.random() < mutation_rate:
                m = (_neighbor_empty_vm(child, safe, conf, rng)
                     if rng.random() < 0.4
                     else _neighbor_move(child, safe, conf, rng))
                if m is not None:
                    child = m
            new_pop.append(child)

        population = new_pop
        fitness = [len(ind) for ind in population]

        cur_best = min(fitness)
        if cur_best < best_cost:
            best_idx = min(range(len(population)), key=lambda i: fitness[i])
            best = deepcopy(population[best_idx])
            best_cost = cur_best

        # Adaptive mutation rate
        if adaptive:
            distinct = len(set(fitness))
            if distinct <= 2:                            # converging
                mutation_rate = min(0.5, mutation_rate * 1.15)
            elif distinct >= pop_size // 2:              # diverse
                mutation_rate = max(0.05, mutation_rate * 0.95)

    return best, best_cost, ""

## 5. Memetic Matheuristic Hybrid

Following Moscato's principle, the hybrid combines a constructive global search with a refinement phase invoking exact methods at small scales. Starting from CSCH-Guided, it identifies dependency components whose VM utilisation indicates room for repacking and submits each one to a small ILP. We evaluate two variants:

- **H-Comp** runs a per-component ILP with a 10-second time budget per component.
- **H-Nest** extends H-Comp with a per-safety-class pass warm-started from the H-Comp solution and given a 60-second budget per class, enabling cross-component reallocation within a safety class.

Both variants delegate the local search to Gurobi configured with the constraint set of the paper's Section III.

### 5.1 Local ILP repack

In [ ]:
def local_ilp_repack(component_apps, safe, conf, *,
                     n_vm_budget=None, dep=None, strict_dependency=False,
                     time_limit=10, verbose=False):
    """Solve the small ILP that re-packs `component_apps` into the fewest VMs.

    Used as the building block by H-Comp, H-Nest, and Path-A ILP.
    """
    if n_vm_budget is None:
        n_vm_budget = sum(1 + int(safe[a]) for a in component_apps)
    is_safe = {a: bool(safe[a] == 1) for a in component_apps}

    m = gp.Model("local_repack", env=GRB_ENV)
    m.setParam("OutputFlag", 1 if verbose else 0)
    m.setParam("TimeLimit", time_limit)

    V = list(range(n_vm_budget))
    x = m.addVars(component_apps, V, vtype=GRB.BINARY, name="x")
    d = m.addVars(V, vtype=GRB.BINARY, name="d")
    s = m.addVars(V, vtype=GRB.BINARY, name="s")

    # (C3) Redundancy
    for a in component_apps:
        m.addConstr(gp.quicksum(x[a, v] for v in V) == (2 if is_safe[a] else 1))
    # (C1) Capacity
    for v in V:
        m.addConstr(gp.quicksum(x[a, v] for a in component_apps) <= MAX_APPS_PER_VM * d[v])
    # (C2) Freedom from interference
    for a in component_apps:
        for v in V:
            if is_safe[a]:
                m.addConstr(x[a, v] <= s[v])
            else:
                m.addConstr(x[a, v] + s[v] <= 1)
    # (C5) Conflict isolation
    for i, a in enumerate(component_apps):
        for b in component_apps[i+1:]:
            if conf[a, b] == 1 or conf[b, a] == 1:
                for v in V:
                    m.addConstr(x[a, v] + x[b, v] <= 1)
    # (C4) Strict dependency (optional)
    if strict_dependency and dep is not None:
        for i, a in enumerate(component_apps):
            for b in component_apps[i+1:]:
                if dep[a, b] == 1 or dep[b, a] == 1:
                    y = m.addVars(V, vtype=GRB.BINARY, name=f"y_{a}_{b}")
                    for v in V:
                        m.addConstr(y[v] <= x[a, v])
                        m.addConstr(y[v] <= x[b, v])
                    m.addConstr(gp.quicksum(y[v] for v in V) >= 1)

    m.setObjective(gp.quicksum(d[v] for v in V), GRB.MINIMIZE)
    m.optimize()

    if m.SolCount == 0:
        return None, 0, f"no_sol_status_{m.status}"

    alloc = {}
    nid = 0
    for v in V:
        if d[v].X < 0.5:
            continue
        apps = [a for a in component_apps if x[a, v].X > 0.5]
        if apps:
            alloc[nid] = {"apps": apps, "safety": bool(s[v].X > 0.5)}
            nid += 1
    smap = {GRB.OPTIMAL: "optimal", GRB.TIME_LIMIT: "time_limit",
            GRB.SUBOPTIMAL: "suboptimal"}
    return alloc, nid, smap.get(m.status, f"status_{m.status}")


def compute_component_stats(allocation, dep, safe):
    """Per-component slack statistics, used by H-Comp to prioritise repacks."""
    comps = components_from_dep(dep)
    n = len(safe)
    placements = {a: [] for a in range(n)}
    for vm, info in allocation.items():
        for a in info["apps"]:
            placements[a].append(vm)
    stats = []
    for cid, C in enumerate(comps):
        n_safe = sum(1 for a in C if safe[a] == 1)
        n_non = len(C) - n_safe
        vms = set()
        for a in C:
            vms.update(placements[a])
        expected = 2 * n_safe + n_non
        ideal = -(-expected // MAX_APPS_PER_VM)
        stats.append({"cid": cid, "apps": list(C), "n_apps": len(C),
                      "n_safe": n_safe, "n_non": n_non,
                      "is_safety": n_safe > 0, "vms": vms, "n_vms": len(vms),
                      "ideal_vms_lb": ideal, "slack": len(vms) - ideal})
    return stats

### 5.2 H-Comp: per-component refinement

In [ ]:
def hybrid_csch_guided(safe, dep, conf, *, time_limit_per_component=10):
    """H-Comp: start from CSCH-Guided, refine each high-slack component via ILP."""
    base, _, reason = csch_guided(safe, dep, conf)
    if base is None:
        return None, 0, reason
    stats = compute_component_stats(base, dep, safe)
    refined = dict(base)
    for cs in sorted(stats, key=lambda s: -s["slack"]):
        if cs["slack"] == 0:
            continue
        new_local, n_local, _ = local_ilp_repack(
            cs["apps"], safe, conf, n_vm_budget=cs["n_vms"],
            time_limit=time_limit_per_component)
        if new_local is None or n_local >= cs["n_vms"]:
            continue
        for vm in cs["vms"]:
            refined.pop(vm, None)
        nid = (max(refined.keys()) + 1) if refined else 0
        for info in new_local.values():
            refined[nid] = info
            nid += 1
    return refined, len(refined), ""

### 5.3 H-Nest: per-component then per-safety-class

After H-Comp, apply a per-safety-class ILP pass that allows applications to move *across* dependency components within the same safety class. This unlocks compactness gains the per-component pass cannot reach.

In [ ]:
def cross_class_refine(safe, dep, conf, allocation, safety_class, *,
                       time_limit=60, mip_start=True):
    """Refine all VMs of a given safety class via a single ILP, warm-started."""
    apps, vms = [], []
    for vm_id, info in allocation.items():
        if info["safety"] == safety_class:
            apps.extend(info["apps"])
            vms.append(vm_id)
    apps = sorted(set(apps))
    if not apps:
        return {}, 0, "empty"

    n_curr = len(vms)
    m = gp.Model("xclass_refine", env=GRB_ENV)
    m.setParam("OutputFlag", 0)
    m.setParam("TimeLimit", time_limit)

    V = list(range(n_curr))
    x = m.addVars(apps, V, vtype=GRB.BINARY, name="x")
    d = m.addVars(V, vtype=GRB.BINARY, name="d")

    for a in apps:
        m.addConstr(gp.quicksum(x[a, v] for v in V) == (2 if safety_class else 1))
    for v in V:
        m.addConstr(gp.quicksum(x[a, v] for a in apps) <= MAX_APPS_PER_VM * d[v])
    for i, a in enumerate(apps):
        for b in apps[i+1:]:
            if conf[a, b] == 1 or conf[b, a] == 1:
                for v in V:
                    m.addConstr(x[a, v] + x[b, v] <= 1)

    if mip_start:
        remap = {old: new for new, old in enumerate(vms)}
        for old in vms:
            for a in allocation[old]["apps"]:
                x[a, remap[old]].Start = 1
            d[remap[old]].Start = 1

    m.setObjective(gp.quicksum(d[v] for v in V), GRB.MINIMIZE)
    m.optimize()

    if m.SolCount == 0:
        return None, 0, f"no_sol_{m.status}"

    new_alloc = {}
    nid = 0
    for v in V:
        if d[v].X < 0.5:
            continue
        apps_here = [a for a in apps if x[a, v].X > 0.5]
        if apps_here:
            new_alloc[nid] = {"apps": apps_here, "safety": safety_class}
            nid += 1
    smap = {GRB.OPTIMAL: "optimal", GRB.TIME_LIMIT: "time_limit",
            GRB.SUBOPTIMAL: "suboptimal"}
    return new_alloc, nid, smap.get(m.status, f"status_{m.status}")


def hybrid_nested(safe, dep, conf, *,
                  time_limit_per_component=10, time_limit_per_class=60):
    """H-Nest: H-Comp followed by a per-safety-class ILP refinement pass."""
    h_comp, _, _ = hybrid_csch_guided(
        safe, dep, conf, time_limit_per_component=time_limit_per_component)
    if h_comp is None:
        return None, 0, "phase_a_failed"
    refined = {}
    nid = 0
    for sc in [True, False]:
        new_local, n_local, _ = cross_class_refine(
            safe, dep, conf, h_comp, sc, time_limit=time_limit_per_class)
        n_orig = sum(1 for i in h_comp.values() if i["safety"] == sc)
        if new_local is None or n_local >= n_orig:
            for vm_id, info in h_comp.items():
                if info["safety"] == sc:
                    refined[nid] = info
                    nid += 1
        else:
            for info in new_local.values():
                refined[nid] = info
                nid += 1
    return refined, len(refined), ""

## 6. Path-A ILP Reference

The Path-A formulation of the paper's Section III applied to the entire instance, used as the exact-method baseline. Solved by Gurobi with a 120-second wall-clock cap.

In [ ]:
def path_a_ilp(safe, dep, conf, *, time_limit=120, strict_dependency=False):
    """Solve the full Path-A ILP for the whole instance."""
    n = len(safe)
    return local_ilp_repack(
        list(range(n)), safe, conf,
        dep=(dep if strict_dependency else None),
        strict_dependency=strict_dependency,
        n_vm_budget=VM_CAP, time_limit=time_limit)

## 7. Adaptive Method Selector

Given an instance and a runtime budget, the selector predicts which method in the portfolio will produce the most compact feasible deployment. Two Random Forest regressors per method predict (i) the VM count and (ii) the runtime; at inference time the selector picks the method with the lowest predicted VM count among those whose predicted runtime fits the budget.

In [ ]:
FEATURE_COLS = [
    "n_apps", "n_safe", "safety_ratio",
    "dep_edges", "dep_density", "n_dep_components",
    "max_dep_comp", "mean_dep_comp", "largest_comp_frac",
    "conf_edges", "conf_density", "mean_conf_deg",
    "max_conf_deg", "conf_clique_lb",
    "mean_total_deg", "max_total_deg",
    "capacity_lb", "per_comp_lb", "lb_gap",
]


def extract_features(safe, dep, conf):
    """Return the 19-dim feature vector used by the selector."""
    n = len(safe)
    n_safe = int(safe.sum())
    n_non = n - n_safe

    dep_sym  = ((dep == 1) | (dep.T == 1)).astype(int);  np.fill_diagonal(dep_sym, 0)
    conf_sym = ((conf == 1) | (conf.T == 1)).astype(int); np.fill_diagonal(conf_sym, 0)
    dep_edges  = int(dep_sym.sum() // 2)
    conf_edges = int(conf_sym.sum() // 2)

    comps = components_from_dep(dep)
    sizes = [len(c) for c in comps]

    deg_dep   = dep_sym.sum(axis=1)
    deg_conf  = conf_sym.sum(axis=1)
    deg_total = deg_dep + deg_conf

    placements = 2 * n_safe + n_non
    cap_lb = -(-placements // MAX_APPS_PER_VM)
    pc_lb = sum(
        -(-(2 * sum(1 for a in C if safe[a] == 1)
            + sum(1 for a in C if safe[a] == 0)) // MAX_APPS_PER_VM)
        for C in comps
    )

    # Conflict-graph clique lower bound (greedy)
    order = sorted(range(n), key=lambda v: -conf_sym[v].sum())
    clique = []
    for v in order:
        if all(conf_sym[v, u] == 1 for u in clique):
            clique.append(v)

    return {
        "n_apps": n, "n_safe": n_safe, "safety_ratio": n_safe / n,
        "dep_edges": dep_edges, "dep_density": dep_edges / max(1, n*(n-1)/2),
        "n_dep_components": len(comps), "max_dep_comp": max(sizes),
        "mean_dep_comp": sum(sizes) / len(comps), "largest_comp_frac": max(sizes) / n,
        "conf_edges": conf_edges, "conf_density": conf_edges / max(1, n*(n-1)/2),
        "mean_conf_deg": float(deg_conf.mean()), "max_conf_deg": int(deg_conf.max()),
        "conf_clique_lb": len(clique),
        "mean_total_deg": float(deg_total.mean()), "max_total_deg": int(deg_total.max()),
        "capacity_lb": cap_lb, "per_comp_lb": pc_lb, "lb_gap": pc_lb - cap_lb,
    }


def train_selector_models(train_df, methods, feature_cols=FEATURE_COLS):
    """Train one (method, target) Random Forest per (method, target) pair."""
    models = {}
    for method in methods:
        sub = train_df[train_df["method"] == method]
        if len(sub) == 0:
            continue
        X = sub[feature_cols].values
        for target in ["vms", "runtime"]:
            y = sub[target].values
            mdl = RandomForestRegressor(
                n_estimators=200, max_depth=10,
                min_samples_leaf=2, random_state=0, n_jobs=-1)
            mdl.fit(X, y)
            models[(method, target)] = mdl
    return models


def select_method(features, time_budget, models, methods):
    """Pick the method with the lowest predicted VM count under the budget."""
    X = np.array([[features[c] for c in FEATURE_COLS]])
    cands = []
    for m in methods:
        if (m, "runtime") not in models:
            continue
        pred_t = models[(m, "runtime")].predict(X)[0]
        pred_v = models[(m, "vms")].predict(X)[0]
        if pred_t <= time_budget:
            cands.append((pred_v, m))
    if not cands:   # Nothing fits the budget — fall back to fastest predicted method
        return min(methods, key=lambda m: models[(m, "runtime")].predict(X)[0])
    return min(cands)[1]

## 8. Synthetic Benchmark Generation

Instances are parameterised by problem size $n$, dependency density $p_{\text{dep}}$, and conflict density $p_{\text{conf}}$. The parameter grid below was selected so that every instance is feasible under the 80-VM cap. At $n = 800$, $p_{\text{conf}}$ around $0.02$–$0.05$ keeps within the cap; tighter grids cause many instances to become infeasible and were filtered out of the training set.

Each instance is run through all seven methods (CSCH-G, SA, TS, GA, H-Comp, H-Nest, ILP-A) with three independent seeds for the stochastic methods. Only instances where every method produces a feasible solution are kept.

In [ ]:
ALL_METHODS = ["CSCH-G", "SA", "TS", "GA", "H-Comp", "H-Nest", "ILP-A"]


def benchmark_instance(safe, dep, conf, *, n_seeds_stochastic=3,
                       time_limit_per_component=10, time_limit_per_class=60,
                       ilp_time_limit=120):
    """Run all seven methods on one instance and record (vms, runtime, feasible)."""
    results = {}

    # CSCH-Guided (deterministic)
    t0 = time.time()
    alloc, n_vm, _ = csch_guided(safe, dep, conf)
    results["CSCH-G"] = {"vms": n_vm, "runtime": time.time() - t0,
                         "feasible": alloc is not None}

    # H-Comp (deterministic)
    t0 = time.time()
    alloc, n_vm, _ = hybrid_csch_guided(
        safe, dep, conf, time_limit_per_component=time_limit_per_component)
    results["H-Comp"] = {"vms": n_vm, "runtime": time.time() - t0,
                         "feasible": alloc is not None}

    # H-Nest (deterministic)
    t0 = time.time()
    alloc, n_vm, _ = hybrid_nested(
        safe, dep, conf,
        time_limit_per_component=time_limit_per_component,
        time_limit_per_class=time_limit_per_class)
    results["H-Nest"] = {"vms": n_vm, "runtime": time.time() - t0,
                         "feasible": alloc is not None}

    # ILP-A (deterministic, time-limited)
    t0 = time.time()
    alloc, n_vm, _ = path_a_ilp(safe, dep, conf, time_limit=ilp_time_limit)
    results["ILP-A"] = {"vms": n_vm, "runtime": time.time() - t0,
                        "feasible": alloc is not None}

    # Stochastic methods: take the best across seeds, runtime is per-seed mean
    for name, fn in [("SA", simulated_annealing),
                     ("TS", tabu_search),
                     ("GA", adaptive_genetic_algorithm)]:
        best_v = None
        total_t = 0.0
        feas = False
        for seed in range(n_seeds_stochastic):
            t0 = time.time()
            alloc, n_vm, _ = fn(safe, dep, conf, seed=seed)
            total_t += time.time() - t0
            if alloc is not None:
                feas = True
                if best_v is None or n_vm < best_v:
                    best_v = n_vm
        results[name] = {"vms": best_v if feas else 0,
                         "runtime": total_t / n_seeds_stochastic,
                         "feasible": feas}
    return results


# Parameter grid: at larger n, p_conf must be smaller to keep instances feasible
PARAM_GRID_BY_SIZE = {
    100: [(0.05, 0.10), (0.10, 0.15), (0.15, 0.20), (0.20, 0.25), (0.30, 0.30)],
    200: [(0.05, 0.08), (0.08, 0.12), (0.12, 0.16), (0.16, 0.20), (0.20, 0.20)],
    400: [(0.04, 0.05), (0.06, 0.08), (0.08, 0.10), (0.10, 0.12), (0.12, 0.12)],
    600: [(0.03, 0.04), (0.04, 0.05), (0.05, 0.06), (0.06, 0.07), (0.07, 0.08)],
    800: [(0.02, 0.03), (0.03, 0.04), (0.04, 0.05), (0.04, 0.06), (0.05, 0.06)],
}

if QUICK_DEMO:
    # Small grid for fast demo (≈10 minutes)
    DEMO_GRID = {100: [(0.10, 0.15)], 200: [(0.08, 0.12)], 400: [(0.06, 0.08)]}
    SEEDS_PER_CELL = 1
    PARAM_GRID = DEMO_GRID
    print("Running QUICK_DEMO (3 instances). Set QUICK_DEMO=False above to reproduce the full paper benchmark.")
else:
    PARAM_GRID = PARAM_GRID_BY_SIZE
    SEEDS_PER_CELL = 2
    print("Running FULL benchmark. Expect 3–4 hours.")

In [ ]:
def run_benchmark(param_grid, seeds_per_cell):
    """Run the full benchmark and return a long-format DataFrame."""
    rows = []
    instances = [(size, p_dep, p_conf, s)
                 for size, grid in param_grid.items()
                 for (p_dep, p_conf) in grid
                 for s in range(seeds_per_cell)]

    for idx, (size, p_dep, p_conf, seed_offset) in enumerate(instances):
        seed = 1000 + size*10 + int(p_dep*100)*100 + int(p_conf*100) + seed_offset
        instance_id = f"n{size}_pd{int(p_dep*100):02d}_pc{int(p_conf*100):02d}_s{seed}"
        print(f"[{idx+1}/{len(instances)}] {instance_id} ...", end=" ", flush=True)
        t0 = time.time()

        safe, dep, conf = generate_instance(size, seed, p_dep=p_dep, p_conf=p_conf)
        feats = extract_features(safe, dep, conf)
        results = benchmark_instance(safe, dep, conf)

        if all(r["feasible"] for r in results.values()):
            for method, perf in results.items():
                rows.append({
                    "instance_id": instance_id,
                    "p_dep_param": p_dep, "p_conf_param": p_conf,
                    **feats,
                    "method": method,
                    "vms": perf["vms"],
                    "runtime": perf["runtime"],
                    "feasible": True,
                })
            print(f"OK [{time.time()-t0:.1f}s]")
        else:
            print(f"INFEASIBLE — skipped [{time.time()-t0:.1f}s]")

    return pd.DataFrame(rows)


# Run benchmark
df = run_benchmark(PARAM_GRID, SEEDS_PER_CELL)
print(f"\nCollected {len(df)} rows across {df['instance_id'].nunique()} instances.")
df.head()

## 9. Results

### 9.1 Compactness and runtime by problem size

Mean VMs and mean runtime per method across problem sizes. CSCH-Guided produces the fastest but least compact deployments; bio-inspired methods achieve substantial compactness gains; H-Nest produces the most compact deployments at small to medium sizes; the time-limited ILP loses its advantage at $n \geq 600$.

In [ ]:
df_feas = df[df["feasible"] == True].copy()

vms_pivot = (df_feas
             .groupby(["n_apps", "method"])["vms"]
             .mean().unstack("method")
             .round(1)
             [ALL_METHODS])
runtime_pivot = (df_feas
                 .groupby(["n_apps", "method"])["runtime"]
                 .mean().unstack("method")
                 .round(2)
                 [ALL_METHODS])

print("Mean VMs used (lower is better)")
print(vms_pivot)
print()
print("Mean runtime in seconds (lower is better)")
print(runtime_pivot)

### 9.2 Cross-validated selector evaluation

Five-fold cross-validation, splitting at the instance level so no instance contributes to both training and test sets. For each runtime budget, we compare the learned selector against five fixed-strategy baselines (Always-X with cascade fallback when X exceeds the budget) and the per-instance oracle.

In [ ]:
def evaluate_selector(df, budgets=(2.0, 10.0, 30.0, 60.0, 120.0, 1e6),
                      n_folds=5, methods=ALL_METHODS):
    instance_ids = df["instance_id"].unique()
    n_folds = min(n_folds, len(instance_ids))
    if n_folds < 2:
        print("Not enough instances for cross-validation; need at least 2.")
        return pd.DataFrame()

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    rows = []

    for fold_idx, (train_i, test_i) in enumerate(kf.split(instance_ids)):
        train_ids = instance_ids[train_i]
        test_ids  = instance_ids[test_i]
        models = train_selector_models(
            df[df["instance_id"].isin(train_ids)], methods=methods)

        for inst_id in test_ids:
            inst_df = df[df["instance_id"] == inst_id]
            features = inst_df.iloc[0][FEATURE_COLS].to_dict()
            actuals = {r["method"]: dict(r) for _, r in inst_df.iterrows()}

            for budget in budgets:
                feasible = {m: a for m, a in actuals.items()
                            if a["runtime"] <= budget}

                def fixed_with_fallback(primary):
                    for m in [primary, "H-Nest", "H-Comp", "TS", "SA", "GA", "CSCH-G"]:
                        if m in feasible:
                            return actuals[m]["vms"]
                    return float("inf")

                pick = select_method(features, budget, models, methods=methods)
                sel_vms = (actuals[pick]["vms"] if pick in feasible
                           else fixed_with_fallback("H-Nest"))
                rows.append({
                    "fold": fold_idx, "instance_id": inst_id,
                    "n_apps": int(features["n_apps"]),
                    "budget": budget,
                    "selector_vms": sel_vms,
                    "always_csch":  actuals["CSCH-G"]["vms"],
                    "always_hnest": fixed_with_fallback("H-Nest"),
                    "always_ilp":   fixed_with_fallback("ILP-A"),
                    "oracle_vms":   min((a["vms"] for a in feasible.values()),
                                        default=float("inf")),
                    "selector_pick": pick,
                })
    return pd.DataFrame(rows)


sel_results = evaluate_selector(df_feas)
if len(sel_results) > 0:
    summary = (sel_results.groupby("budget")
               .agg(selector=("selector_vms", "mean"),
                    csch=("always_csch", "mean"),
                    hnest=("always_hnest", "mean"),
                    ilp=("always_ilp", "mean"),
                    oracle=("oracle_vms", "mean"))
               .round(2))
    print("Mean VMs by runtime budget")
    print(summary)

    sel_results["regret"] = sel_results["selector_vms"] - sel_results["oracle_vms"]
    print("\nRegret vs oracle (mean)")
    print(sel_results.groupby("budget")["regret"].mean().round(2))

    print("\nSelector pick distribution")
    print(sel_results["selector_pick"].value_counts())

### 9.3 Visualisation

Two-panel figure: (a) compactness as a function of $n$, (b) runtime as a function of $n$. This is Figure 1 of the paper.

In [ ]:
if len(df_feas) > 0:
    method_styles = {
        "CSCH-G": dict(color="#666666", marker="o", linestyle=":"),
        "SA":     dict(color="#1f77b4", marker="s", linestyle="-"),
        "TS":     dict(color="#ff7f0e", marker="D", linestyle="-"),
        "GA":     dict(color="#2ca02c", marker="^", linestyle="-"),
        "H-Comp": dict(color="#d62728", marker="v", linestyle="--"),
        "H-Nest": dict(color="#9467bd", marker="P", linestyle="--"),
        "ILP-A":  dict(color="#000000", marker="X", linestyle=":"),
    }

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))

    for method in ALL_METHODS:
        sub = (df_feas[df_feas["method"] == method]
               .groupby("n_apps")
               .agg(vms=("vms", "mean"), runtime=("runtime", "mean"))
               .reset_index())
        if len(sub) == 0:
            continue
        style = method_styles[method]
        for ax, col in [(axes[0], "vms"), (axes[1], "runtime")]:
            ax.plot(sub["n_apps"], sub[col].clip(lower=0.01) if col == "runtime" else sub[col],
                    color=style["color"], marker=style["marker"],
                    linestyle=style["linestyle"], label=method,
                    linewidth=1.3, markersize=6, alpha=0.9)

    axes[0].set_xlabel("Number of applications $n$")
    axes[0].set_ylabel("Mean VMs used")
    axes[0].set_title("(a) Compactness")
    axes[0].grid(True, alpha=0.3)

    axes[1].set_xlabel("Number of applications $n$")
    axes[1].set_ylabel("Mean runtime (s, log scale)")
    axes[1].set_yscale("log")
    axes[1].set_title("(b) Runtime")
    axes[1].grid(True, alpha=0.3, which="both")

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center",
               bbox_to_anchor=(0.5, -0.02), ncol=7, fontsize=9, frameon=False)
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.20)
    plt.show()

## 10. Discussion

### Summary of findings

The benchmark sweep confirms three observations that drove the paper's main claims.

**Bio-inspired methods close the gap with the exact ILP at scale.** At $n = 800$, Simulated Annealing, Tabu Search, and the Genetic Algorithm all reach 61.0 VMs against the time-limited ILP's 63.9, while finishing one to two orders of magnitude faster than ILP-A on every problem size we tested. SA in particular completes $n = 800$ in under 6 seconds (per-seed mean) compared with ILP-A's ≈131 seconds — a 22× runtime advantage at comparable or better compactness. The shared adaptive control mechanism (cooling rate for SA, tabu length for TS, mutation rate for GA) keeps each search from stagnating, which matters most on the larger and tighter instances.

**The memetic hybrid wins at small to medium scale.** H-Nest produces the most compact deployments at $n \leq 400$, beating every other method including Path-A ILP at these sizes. Its per-component decomposition keeps each ILP solvable within reasonable time, whereas the direct Path-A ILP on the full instance exceeds budget on most non-trivial instances without proving optimality. At $n \geq 600$ the per-class ILP inside H-Nest itself begins to time out, which is exactly where the bio-inspired methods take over.

**No single method dominates, motivating the selector.** The pick distribution from the cross-validated selector (≈36% H-Nest, ≈22% CSCH-G, ≈21% SA, ≈13% TS) shows that the best choice is genuinely instance-dependent. The selector tracks the per-instance oracle within an average regret of less than one VM across all runtime budgets, while delivering the same level of compactness that the strongest fixed strategy achieves only with cascade fallback to ground-truth runtimes — a luxury not available in real deployment.

### Strengths

- **Fair single-execution runtime accounting.** Stochastic methods report per-seed mean runtime, putting them on the same footing as deterministic methods. This avoids the common metaheuristic-paper trap of comparing best-of-N against a single ILP run.
- **Constraint-correct verification.** Every reported allocation is run through an explicit verifier covering all five constraint families before being recorded as feasible.
- **Realistic budget-aware selection.** The selector receives only an instance feature vector and a runtime budget; it does not see ground-truth runtimes. This matches what a real planner would have available at design time.
- **Reproducibility.** The benchmark is fully synthetic, parameterised by seed, and runs end-to-end in this notebook with no external data.

### Limitations

- **Uniform per-application demand.** All applications use $0.05$ cores and $50$ MB. Real automotive workloads vary by orders of magnitude across ADAS, perception, infotainment, and body control. Heterogeneous demands would change the bin-packing structure and likely shift the relative ordering of methods.
- **Operational-feasibility relaxation.** The dependency constraint (C4) is enforced operationally (dependent applications share at least one component) rather than per-edge. This is consistent with prior work but leaves the strict-dependency variant unexplored.
- **Limited evaluation pool.** 48 instances is enough for cross-validated comparison but is a thin training set for the selector; pick stability would tighten with larger datasets.
- **Fixed VM size.** Path-A's homogeneous 1-core/96-GB VM model is one of two paths in the literature; the variable-VM-size path is not addressed here.
- **Synthetic generator.** Edge probabilities are tuned to keep the 80-VM cap reachable, which prunes the densest instances. Industrial workloads may exhibit structure (clustering, dependency hierarchies) the uniform random model does not capture.

### Connecting results to method design

Each design choice in the paper has a visible footprint in the results. The **adaptive cooling schedule** of SA explains why a single SA run reaches the same compactness as TS and GA at large $n$ in a fraction of the time — the cooling rate ramps up as acceptance drops and the search converges fast. The **reactive tabu length** of TS prevents the diversification stalls that fixed-tabu-length implementations exhibit on dense conflict graphs around $n = 200$–$400$. The **component-wise crossover** of GA gives feasibility-preservation for free, removing the repair overhead that would otherwise dominate runtime; this is what keeps GA competitive with SA on quality despite a much smaller iteration budget. The **per-component decomposition** of H-Comp turns an intractable global ILP into a sequence of small, fast subproblems, and the **per-class refinement** of H-Nest recovers the cross-component compactness that the per-component pass cannot reach. Finally, the **runtime-aware selector** translates these per-method profiles into a single deployable rule that adapts to whatever runtime budget the integrator can afford.

### Future work

Extending the framework toward operational adoption involves replacing uniform per-application demands with realistic workload profiles, developing benchmark generators that yield instances feasible under the strict per-edge interpretation of constraint (C4), extending the selector to runtime decisions during over-the-air updates and hardware fault recovery, relaxing the fixed-VM-size assumption, and validating on real automotive HPC platforms with ISO 26262 certification considerations.